[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/24_rope.ipynb)

# 🔴 Hard: Rotary Position Embedding (RoPE)

Implement **RoPE** — the position encoding used in LLaMA, GPT-NeoX, and most modern LLMs.

### Signature
```python
def apply_rope(q: Tensor, k: Tensor) -> tuple[Tensor, Tensor]:
    # q, k: (B, S, D) where D is even
    # Returns rotated (q, k) with same shape
```

### Key Idea
Split each vector into consecutive pairs. Rotate each pair by `θ = pos / 10000^(2i/D)`:
```
[x_0, x_1] → [x_0*cosθ - x_1*sinθ, x_0*sinθ + x_1*cosθ]
```
This makes `dot(q_rot[i], k_rot[j])` depend only on `i - j` (relative position).

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 3.1 MB/s eta 0:00:00


In [2]:
import torch
import math

In [18]:

def apply_rope(q, k):
    """
    Q: (B, S, D)
    K: (B, S, D)

    D even
    results: same shape (Q, K)
    """

    _, S, D = k.shape
    halfD = D // 2

    pos = torch.arange(S).unsqueeze(1) #(S, 1)
    i = torch.arange(halfD).unsqueeze(0) #(1, halfD)

    theta =  pos / (10000 ** ((2 * i) / D)) #(S, D//2)

    #repeat frequencies for the two halves
    theta = torch.cat([theta, theta], dim = -1) #(S, D)

    q1 = q[:, :, :halfD]
    q2 = q[:, :, halfD:]

    k1 = k[:, :, :halfD]
    k2 = k[:, :, halfD:]


    q_rotated = torch.cat([-q2, q1], dim = -1)
    k_rotated = torch.cat([-k2, k1], dim = -1)

    #element-wise mul
    q_rope = q * torch.cos(theta) + q_rotated * torch.sin(theta)
    k_rope = k * torch.cos(theta) + k_rotated * torch.sin(theta)

    return (q_rope, k_rope)





In [19]:
# 🧪 Debug
q = torch.randn(1, 8, 16)
k = torch.randn(1, 8, 16)
qr, kr = apply_rope(q, k)
print('Shape preserved:', qr.shape == q.shape)
print('Norm preserved:', torch.allclose(q.norm(dim=-1), qr.norm(dim=-1), atol=1e-4))

Shape preserved: True
Norm preserved: True


In [20]:
# ✅ SUBMIT
from torch_judge import check
check('rope')


🧪 Testing: Rotary Position Embedding (RoPE) (Hard)
──────────────────────────────────────────────────
  ✅ [1/4] Output shapes (1.7ms)
  ✅ [2/4] Preserves norm (3.7ms)
  ✅ [3/4] Relative position property (2.5ms)
  ✅ [4/4] Gradient flow (1.0ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (8.8ms total)
  Progress saved. Run status() to see your dashboard.

